In [2]:
import pandas as pd
import micropip
await micropip.install("openpyxl")

acc = pd.read_csv("Accidents.csv")
par = pd.read_csv("Parties.csv")
veh = pd.read_csv("Vehicles.csv")
veh2 = pd.read_excel("Vehicles2.xlsx")

# 1) الأحجام
print(len(acc), len(par), len(veh), len(veh2))

# 2) الحوادث اللي ناقصة من Vehicles2
missing = set(acc["رقم الحادث"]) - set(veh2["رقم الحادث"])
print("ناقصة من Vehicles2:", missing)

# 3) فرق الصفوف بين الملفين
print(veh.groupby("رقم الحادث").size().sum(),
      veh2.groupby("رقم الحادث").size().sum())
print(veh2["vehicle_id"].value_counts().head())

# 4) تصحيح أسماء الإحداثيات
acc = acc.rename(columns={"Xcoordinate": "lat", "Ycoordinate": "lon"})
print(acc[["lat", "lon"]].describe())

520 1004 912 942
ناقصة من Vehicles2: {6003572942, 6003412901}
912 942
vehicle_id
1    517
2    398
3     19
4      6
5      2
Name: count, dtype: int64
              lat         lon
count  520.000000  520.000000
mean    26.442762   49.672516
std      0.698313    1.058487
min     24.252185   45.522709
25%     26.302146   49.764233
50%     26.415040   50.027392
75%     26.561643   50.084997
max     28.470347   51.302501


In [3]:
# 1) نقارن عدد المركبات لكل حادث بين الملفين
c1 = veh.groupby("رقم الحادث").size().rename("v1")
c2 = veh2.groupby("رقم الحادث").size().rename("v2")
cmp = pd.concat([c1, c2], axis=1).fillna(0).astype(int)
diff = cmp[cmp.v1 != cmp.v2]
print("حوادث فيها فرق:", len(diff))
print(diff.head(10))
print((diff.v2 - diff.v1).value_counts())

# 2) الحادثين الناقصين كم مركبة لهم في Vehicles
print(cmp.loc[[6003572942, 6003412901]])

# 3) هل الإحداثيات الشاذة هي حفر الباطن؟
print(acc.groupby("المدينة")[["lat", "lon"]].agg(["min", "max"]).round(2))

حوادث فيها فرق: 31
            v1  v2
رقم الحادث        
100442923    1   2
100444322    1   2
100640329    1   2
6002048237   2   4
6002064713   1   2
6002066345   1   2
6002127709   1   2
6002981246   1   2
6003121072   2   3
6003147272   3   4
 1    24
 2     2
-1     2
 3     2
-2     1
Name: count, dtype: int64
            v1  v2
رقم الحادث        
6003572942   2   0
6003412901   1   0
                 lat           lon       
                 min    max    min    max
المدينة                                  
الأحســــاء    24.95  26.17  48.07  50.30
الجبيل         26.85  27.38  48.99  49.82
الخبر          26.16  26.39  50.15  50.23
الخفجي         28.41  28.47  48.46  48.48
الدمام         26.29  26.50  49.78  50.19
الظهران        26.15  26.38  49.92  50.18
العديد         24.25  24.69  50.73  51.30
القطيف         26.47  26.67  49.80  50.08
النعيريه       26.98  27.47  48.39  48.51
بقيق           26.02  26.02  49.74  49.74
جسر الملك فهد  26.22  26.22  50.24  50.24
حفر الباطن     27.

In [4]:
drivers = (par[par["نوع الطرف"].isin(["سائق", "سائق هارب"])]
           .groupby("رقم الحادث").size().rename("drivers"))

chk = pd.concat([c1, c2, drivers], axis=1).fillna(0).astype(int)
d = chk[chk.v1 != chk.v2].copy()
d["v1_ok"] = d.v1 == d.drivers
d["v2_ok"] = d.v2 == d.drivers
print(d[["v1_ok", "v2_ok"]].sum())
print(d.head(10))

# مثال: صفوف حادث واحد في الملفين
ex = 100442923
print(veh[veh["رقم الحادث"] == ex])
print(veh2[veh2["رقم الحادث"] == ex])

v1_ok     4
v2_ok    23
dtype: int64
            v1  v2  drivers  v1_ok  v2_ok
رقم الحادث                               
100442923    1   2        2  False   True
100444322    1   2        2  False   True
100640329    1   2        2  False   True
6002048237   2   4        4  False   True
6002064713   1   2        2  False   True
6002066345   1   2        2  False   True
6002127709   1   2        2  False   True
6002981246   1   2        2  False   True
6003121072   2   3        2   True  False
6003147272   3   4        4  False   True
   رقم الحادث  نوع المركبة             تسجيل المركبة
9   100442923  دراجة نارية  المملكة العربية السعودية
    vehicle_id  رقم الحادث             تسجيل المركبة  نوع المركبة الموديل  \
9            1   100442923  المملكة العربية السعودية  دراجة نارية    أخرى   
10           2   100442923  المملكة العربية السعودية  دراجة نارية    أخرى   

               الشركة  
9   أخرى يذكر بالملخص  
10  أخرى يذكر بالملخص  


In [5]:
bad = d[~d.v2_ok]
print(bad)

# الأطراف في هذي الحوادث
print(par[par["رقم الحادث"].isin(bad.index)]
      .sort_values("رقم الحادث")[["رقم الحادث", "نوع الطرف", "العمر"]])

            v1  v2  drivers  v1_ok  v2_ok
رقم الحادث                               
6003121072   2   3        2   True  False
6003148437   2   1        2   True  False
6003412901   1   0        1   True  False
6003572942   2   0        3  False  False
6003845444   2   5        3  False  False
6003847531   2   3        1  False  False
6004137595   3   4        2  False  False
6004355054   1   2        1   True  False
     رقم الحادث نوع الطرف  العمر
313  6003121072      راكب   30.0
314  6003121072      سائق   34.0
315  6003121072      سائق   34.0
326  6003148437      سائق   22.0
327  6003148437      سائق   22.0
464  6003412901      سائق   24.0
537  6003572942      سائق   36.0
538  6003572942      سائق   44.0
536  6003572942      سائق   29.0
753  6003845444      سائق   18.0
754  6003845444      سائق   26.0
755  6003845444      سائق   40.0
756  6003847531       NaN   21.0
757  6003847531       NaN   27.0
758  6003847531      سائق   40.0
873  6004137595      سائق   24.0
872  6004137595    

In [6]:
# 1) المركبات النهائية
miss = [6003572942, 6003412901]
extra = veh[veh["رقم الحادث"].isin(miss)].copy()
extra["vehicle_id"] = extra.groupby("رقم الحادث").cumcount() + 1
veh_final = pd.concat([veh2, extra], ignore_index=True)

# 2) علامة الحوادث المشكوك فيها
flag = [6003121072, 6003148437, 6003572942, 6003845444,
        6003847531, 6004137595, 6004355054]
acc["vehicle_check"] = acc["رقم الحادث"].isin(flag)

# 3) جدول موحد على مستوى الحادث
pa = par.groupby("رقم الحادث").agg(
    parties=("العمر", "size"),
    min_age=("العمر", "min"),
    max_age=("العمر", "max"))
vs = veh_final.groupby("رقم الحادث").size().rename("vehicles")

master = (acc.merge(pa, on="رقم الحادث", how="left")
             .merge(vs, on="رقم الحادث", how="left"))
master["is_fatal"] = master["جسامة الحادث"].eq("حادث وفاة")

print(master.shape)
print(master.isna().sum()[lambda s: s > 0])
print(par.isna().sum())

(520, 17)
Series([], dtype: int64)
رقم الحادث        0
العمر             8
الجنس             6
الحاله الصحية     8
الجنسية           9
نوع الطرف        10
dtype: int64


In [7]:
import numpy as np

# 1) تنظيف الأطراف
par_c = par.copy()
par_c["العمر"] = par_c["العمر"].replace(0, np.nan)
for c in ["نوع الطرف", "الجنس", "الحاله الصحية", "الجنسية"]:
    par_c[c] = par_c[c].fillna("غير معروف")

# 2) إعادة حساب العمر
age = par_c.groupby("رقم الحادث")["العمر"].agg(min_age="min", max_age="max")
master = master.drop(columns=["min_age", "max_age"]).merge(
    age, on="رقم الحادث", how="left")

# 3) فحص تحقق: الوفيات في Parties مقابل جسامة الحادث
death = par_c["الحاله الصحية"].str.contains("وفاة")
fatal_ids = set(par_c.loc[death, "رقم الحادث"])
print("حوادث فيها وفاة في Parties:", len(fatal_ids))
print("حوادث وفاة في Accidents:", master["is_fatal"].sum())
print("غير متطابقة:", len(fatal_ids ^ set(master.loc[master.is_fatal, "رقم الحادث"])))

# 4) أول تحليل: نسبة الوفاة حسب كل عامل (نستبعد المجموعات الصغيرة)
def rate(col, min_n=15):
    t = master.groupby(col)["is_fatal"].agg(["size", "mean"])
    t = t[t["size"] >= min_n].sort_values("mean", ascending=False)
    t["mean"] = (t["mean"] * 100).round(1)
    return t.rename(columns={"size": "حوادث", "mean": "% وفاة"})

for col in ["السنة", "المدينة", "سبب الحادث", "نوع الحادث"]:
    print(rate(col), "\n")

print(master["الساعة"].min(), master["الساعة"].max())

حوادث فيها وفاة في Parties: 76
حوادث وفاة في Accidents: 76
غير متطابقة: 0
       حوادث  % وفاة
السنة               
2023     137    16.8
2024     184    14.1
2025     199    13.6 

             حوادث  % وفاة
المدينة                   
الأحســــاء     78    38.5
الخبر           69    13.0
حفر الباطن      37    10.8
الدمام         126     9.5
الظهران         40     7.5
القطيف         148     7.4 

                       حوادث  % وفاة
سبب الحادث                          
عدم ترك مسافة كافية       46    19.6
تجاوز الإشارة الحمراء     26    15.4
إنحراف مفاجئ             152    14.5
إنشغال عن القيادة         60    13.3
عكس اتجاه السير           23    13.0
مخالفة الافضلية           88    12.5
أخرى "تذكر بالملخص"       45    11.1
صدم من الخلف              17     5.9 

                  حوادث  % وفاة
نوع الحادث                     
صدم مركبة متحركة     18    27.8
صدم جسم ثابت اخر     33    27.3
دهس                  16    18.8
إنقلاب مركبة         45    15.6
صدم دراجة نارية     382    12.8 

0 2

In [8]:
from scipy.stats import fisher_exact

def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z**2 / n
    c = p + z**2 / (2 * n)
    m = z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5)
    return round((c - m) / d * 100, 1), round((c + m) / d * 100, 1)

# 1) الأحساء مقابل الباقي
ah = master["المدينة"].str.contains("الأح")
tab = pd.crosstab(ah, master["is_fatal"])
print(tab)
print("p-value:", fisher_exact(tab)[1])
print("هامش الأحساء:", wilson(tab.loc[True, True], tab.loc[True].sum()))
print("هامش الباقي:", wilson(tab.loc[False, True], tab.loc[False].sum()))

# 2) هل الأحساء تختلف في نوع الحادث والسبب؟
for col in ["نوع الحادث", "سبب الحادث"]:
    print(pd.crosstab(master[col], ah, normalize="columns").round(2).head(6), "\n")

# 3) الفترة من اليوم
master["فترة"] = pd.cut(master["الساعة"], [-1, 5, 11, 17, 23],
                        labels=["0-5", "6-11", "12-17", "18-23"])
print(master.groupby("فترة", observed=True)["is_fatal"].agg(["size", "mean"]).round(3))

is_fatal  False  True 
المدينة               
False       396     46
True         48     30
p-value: 7.581346060554067e-09
هامش الأحساء: (np.float64(28.4), np.float64(49.6))
هامش الباقي: (np.float64(7.9), np.float64(13.6))
المدينة           False  True 
نوع الحادث                    
إنقلاب مركبة       0.08   0.12
دهس                0.02   0.06
صدم جسم ثابت اخر   0.06   0.09
صدم حاجز جانبي     0.01   0.00
صدم حيوان          0.00   0.00
صدم دراجة نارية    0.75   0.67 

المدينة              False  True 
سبب الحادث                       
أخرى "تذكر بالملخص"   0.10   0.01
إنحراف مفاجئ          0.31   0.22
إنشغال عن القيادة     0.12   0.09
استدارة               0.00   0.00
الرجوع للخلف          0.02   0.05
السير في وسط الشارع   0.00   0.01 

       size   mean
فترة              
0-5     130  0.169
6-11     53  0.132
12-17   137  0.117
18-23   200  0.155


In [9]:
%pip install statsmodels

In [10]:
import numpy as np
import statsmodels.formula.api as smf

m_ = master.copy()
m_["ahsa"] = ah.astype(int)
m_["fatal"] = m_["is_fatal"].astype(int)
m_["year"] = m_["السنة"]
m_["night"] = m_["الساعة"].isin([*range(18, 24), *range(0, 6)]).astype(int)
m_["hit_moto"] = (m_["نوع الحادث"] == "صدم دراجة نارية").astype(int)

# 1) هل الفرق ثابت كل سنة؟
print(m_.groupby(["year", "ahsa"])["fatal"].agg(["size", "sum", "mean"]).round(3))

# 2) انحدار لوجستي: أثر الأحساء بعد ضبط العوامل
mod = smf.logit("fatal ~ ahsa + night + hit_moto + C(year)", data=m_).fit(disp=0)
ci = mod.conf_int()
print(pd.DataFrame({"OR": np.exp(mod.params),
                    "low": np.exp(ci[0]), "high": np.exp(ci[1]),
                    "p": mod.pvalues}).round(3))

           size  sum   mean
year ahsa                  
2023 0      119   16  0.134
     1       18    7  0.389
2024 0      159   13  0.082
     1       25   13  0.520
2025 0      164   17  0.104
     1       35   10  0.286
                    OR    low   high      p
Intercept        0.155  0.073  0.328  0.000
C(year)[T.2024]  0.775  0.409  1.471  0.436
C(year)[T.2025]  0.675  0.356  1.278  0.227
ahsa             5.510  3.152  9.633  0.000
night            1.427  0.821  2.480  0.207
hit_moto         0.656  0.381  1.130  0.129


In [11]:
out = master.rename(columns={
    "المدينة": "city", "السنة": "year", "الشهر": "month", "الساعة": "hour",
    "سبب الحادث": "cause", "نوع الحادث": "type"})[
    ["year", "month", "hour", "city", "cause", "type",
     "is_fatal", "lat", "lon", "parties", "vehicles"]]
out.to_json("accidents_clean.json", orient="records", force_ascii=False)
print(out.shape, out.is_fatal.sum())

(520, 11) 76
